In [1]:
# common imports
import os
import time
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, precision_score, recall_score

# audio libraries
import librosa
import audiomentations
import IPython.display as ipd

# modeling imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset
from torchvision import transforms
import torchvision.models as models

# documentation
import numpy.typing as npt
from typing import Optional, Union

In [2]:
# where your data is
meta = pd.read_csv('../data/combined.metadata.split.timeslices.csv')
onehot = pd.read_csv('../data/onehot_labels.csv')

# global variables for file paths
highest_folder = '/mnt/d/acoustics-data'
map_collection = {
    'iNat': 'birdclef-2026/train_audio',
    'XC': 'birdclef-2026/train_audio',
    'esc': 'ESC-50-master/ESC-50-master/audio',
    'arca23k': 'ARCA23K/ARCA23K.audio/ARCA23K.audio',
    'urban8k': 'Urban8K/audio',
    'dbr': 'dbr-dataset/dataset/dograin',
    'freesound': 'freesound/outside',
    'random_noise': 'random-noise',
}

# global variables for mel spectrogram
height = 224 # n_mels
width = 224
fmin = 0
fmax = 16000
duration = 5

In [3]:

def reformat_image(input_tensor: torch.Tensor, 
                   image_size: tuple = (224,224), 
                   channel_size: int =3,
                   ) -> torch.Tensor:
    '''Prepare spectrogram for preexisting models
    
    Parameters
    ----------
    input_tensor : torch.Tensor
        Shape must be (B,C,H,W), (C,H,W), or (H,W)
    image_size : tuple
        Desired size (H*,W*)
    channel_size : int
        3 for RGB images

    Returns
    -------
    torch.Tensor
        Normalized and resized image(s)
    '''

    # insist on floating point precision
    input_tensor = input_tensor.detach().clone().to(dtype=torch.float32)

    # add batch dimension if single
    if len(input_tensor.shape) < 3:
        input_tensor = input_tensor.unsqueeze(0)
    if len(input_tensor.shape) < 4:
        input_tensor = input_tensor.unsqueeze(0)

    # convert to rgb in most cases
    if input_tensor.shape[1] != channel_size:
        input_tensor = input_tensor.repeat(1, channel_size, 1, 1)

    # normalize
    if input_tensor.max() > 1.0:
        input_tensor /= 255.0
    if channel_size == 3:
        normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    elif channel_size == 1:
        normalize = transforms.Normalize(mean=[0.449], std=[0.226])
    else:
        raise ValueError("Channel size is not 1 (grayscale) or 3 (RGB)")
    
    if input_tensor.shape[-2:] != image_size:
        preproc = transforms.Compose([
            transforms.Resize(image_size, interpolation=transforms.InterpolationMode.BICUBIC),
            normalize,
        ])
    else:
        preproc = transforms.Compose([
            normalize, 
        ])

    input_final = preproc(input_tensor)

    return input_final

def clean_row(row: pd.Series, ):
    '''Subset the useful info in each row'''
    row = row[
        [
            'primary_label',
            'index_label',
            'common_name',
            'sampling_rate_hz',
            'start_seconds',
            'end_seconds',
            'filename',
            'collection',
            'latitude',
            'longitude',
            'class_name',
            'dataset'
        ]
    ]
    return dict(row)

def get_audio(row: dict, ):
    '''Load in the audio for a row'''
    fname = f'{highest_folder}/{map_collection[row['collection']]}/{row['filename']}'
    y, sr = librosa.load(fname, 
                         sr=row['sampling_rate_hz'],
                         duration=duration,
                         offset=row['start_seconds'],
                         )
    return y, sr

def get_mel(y: npt.NDArray, sr: int, ):
    '''Get a melspectrogram image'''
    l = sr * duration
    h = l / (width - 1)
    hupp = math.ceil(h) # calculate hop length
    x = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        hop_length=hupp,
        n_mels=height,
        fmin=fmin,
        fmax=fmax
    )
    if x.shape != (width, height):
        # force the array dim
        x = librosa.util.fix_length(x, size=width, axis=0)
        x = librosa.util.fix_length(x, size=height, axis=1)
    return x, hupp

In [43]:
class AudioDataset(Dataset):
    def __init__(self, df, num_classes):
        """
        Parameters
        ----------
            df : pd.DataFrame
            num_classes : int
        """
        self.df = df.reset_index(drop=True) # Ensure indices are sequential
        self.num_classes = num_classes

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # 1. Get the row data
        row = self.df.iloc[idx]
        row = clean_row(row)

        # process the audio
        audio, sr = get_audio(row)
        x, h = get_mel(audio, sr)
        # x_db = librosa.power_to_db(x, ref=np.max)
        
        # 4. Convert to tensor
        x_tensor = torch.from_numpy(x).float()
        x_tensor = x_tensor.unsqueeze(0)
        y = row['index_label']
        y_tensor = torch.tensor(y)
        y_one_hot = F.one_hot(y_tensor, self.num_classes).float()
            
        return x_tensor, y_one_hot

In [5]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes, channel_size):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(channel_size, 16, kernel_size=3, padding=1),   # 224x224
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 112x112

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 56x56

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 28x28

            nn.AdaptiveAvgPool2d((1, 1))                  # 1x1
        )
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

In [6]:
# Build explicit label maps from onehot (index <-> primary_label)
label_df = onehot[['index', 'primary_label']].drop_duplicates().sort_values('index')
label_to_index = dict(zip(label_df['primary_label'], label_df['index']))
num_classes = len(label_to_index)
print(f"num classes: {len(label_to_index)}")
print("example:", "banana ->", label_to_index.get("banana"))

num classes: 238
example: banana -> 80


In [7]:
# for testing
meta = meta[meta.collection.isin(['XC','iNat'])]
meta.reset_index(inplace=True, drop=True)

In [8]:
# prepare the datasets
train_data = meta[meta['dataset']=='train'].copy()
val_data = meta[meta['dataset']=='validate'].copy()
test_data = meta[meta['dataset']=='test'].copy()
train_data.reset_index(inplace=True, drop=True)
val_data.reset_index(inplace=True, drop=True)
test_data.reset_index(inplace=True, drop=True)
# Optional: apply to your splits
train_data.loc[:, 'index_label'] = train_data['primary_label'].map(label_to_index)
val_data.loc[:, 'index_label'] = val_data['primary_label'].map(label_to_index)
test_data.loc[:, 'index_label'] = test_data['primary_label'].map(label_to_index)

In [9]:
# listen to an example audio
idx = 1000
row = clean_row(train_data.iloc[idx])
y, sr = get_audio(row)
x, h = get_mel(y, sr)
print(row['common_name'])
ipd.Audio(y, rate=sr)

Snouted Tree Frog


In [61]:
# smaller subset: 8 batches of 32 samples = 256
n_samples = 4 * 8

# sample from test split (use .head(n_samples) instead of .sample(...) if you want deterministic order)
train_data_small = train_data.sample(n=min(n_samples, len(train_data)), random_state=42).reset_index(drop=True)
val_data_small = val_data.sample(n=min(n_samples, len(val_data)), random_state=42).reset_index(drop=True)

# optional: overwrite active dataset/dataloader to use the small subset
dataset_train = AudioDataset(train_data_small, num_classes)
dataloader_train = DataLoader(dataset_train, batch_size=8, num_workers=0, shuffle=False)
dataset_val = AudioDataset(val_data_small, num_classes)
dataloader_val = DataLoader(dataset_val, batch_size=8, num_workers=0, shuffle=False)

print("test_data_small:", len(train_data_small))
print("num batches:", len(dataloader_train))

test_data_small: 32
num batches: 4


In [ ]:
# weights = models.EfficientNet_B0_Weights.DEFAULT
# model = models.efficientnet_b0(weights=weights)

# in_features = model.classifier[1].in_features
# model.classifier = nn.Sequential(
#     nn.Dropout(p=0.2, inplace=True),
#     nn.Linear(in_features, num_classes),
# )

# criterion = nn.BCEWithLogitsLoss()
# optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [56]:

# use this to see if code will run
model = SimpleCNN(num_classes=num_classes, channel_size=1)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [57]:
# total and trainable parameter counts
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Non-trainable parameters: {total_params - trainable_params:,}")

Total parameters: 38,766
Trainable parameters: 38,766
Non-trainable parameters: 0


In [62]:
num_epochs = 2

ft = open('training_loss.txt','w')
fv = open('validate_loss.txt','w')

for _ in range(num_epochs):

    # learning
    model.train()
    loss_total = 0
    itr = 0
    curr_time = time.time()
    for X, Y in dataloader_train:
        itr += 1
        optimizer.zero_grad()
        Z = reformat_image(X, 
                           image_size=(224,224), 
                           channel_size=1
                           )
        logits = model(Z)
        loss = criterion(logits, Y)
        loss_total += loss
        loss.backward()
        optimizer.step()
    print(f'Epoch {_} took {time.time() - curr_time:4f} seconds')
    print('')

    # inference
    model.eval()
    with torch.no_grad():

        # training loss tracking
        loss_total = 0
        itr = 0
        for X, Y in dataloader_train:
            itr += 1
            Z = reformat_image(X, 
                           image_size=(224,224), 
                           channel_size=1
                           )
            logits = model(Z)
            loss = criterion(logits, Y)
            loss_total += loss
        print(f'The training loss at epoch {_} is {loss_total / itr:6f}')
        ft.write(f"{loss_total / itr:6f}\n")

        # validation loss tracking
        loss_total = 0
        itr = 0
        for X, Y in dataloader_val:
            itr += 1
            Z = reformat_image(X, 
                           image_size=(224,224), 
                           channel_size=1
                           )
            logits = model(Z)
            loss = criterion(logits, Y)
            loss_total += loss
        print(f'The validation loss at epoch {_} is {loss_total / itr:6f}')
        fv.write(f"{loss_total / itr:6f}\n")
    
    print('')

ft.close()
fv.close()

Epoch 0 took 1.638861 seconds

The training loss at epoch 0 is 0.070757
The validation loss at epoch 0 is 0.072951

Epoch 1 took 2.397427 seconds

The training loss at epoch 1 is 0.042239
The validation loss at epoch 1 is 0.046436

